# Introduction to Polars

[Polars](https://pola.rs/) is a data processing library for Python and Rust designed to work with tabular data. It is similar to pandas, but it is faster, more memory-efficient, and better at handling large datasets.

In this notebook, you will use Polars to open a local Parquet file, inspect the taxi data, and build a few simple summaries. It is the first step in the batch-processing path and focuses on reading and transforming data locally.

**Learning goals:**
- Load Parquet data with Polars.
- Inspect a schema and a few rows.
- Filter, sort and aggregate taxi rides locally.

Before you run this notebook, download at least one Parquet file with:

```bash
uv run python src/data_ingestion.py --year 2025 --months 1 --output-dir data/raw
```


## Outline

1. Load one Parquet file from `data/raw/`.
2. Inspect the columns and data types.
3. Filter and sort rides.
4. Build a small grouped summary.
5. Try a short exercise.


In [ ]:
from pathlib import Path

import polars as pl

DATA_PATH = Path("../data/raw/yellow_tripdata_2025-01.parquet")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing {DATA_PATH}. Run src/data_ingestion.py first.")

rides = pl.read_parquet(DATA_PATH)
print(f"Loaded {rides.height:,} rows and {rides.width} columns")
rides.schema

## Step 1 - Preview the dataset

Start small. A quick preview tells us what kind of fields we have before we build any transformations.


In [ ]:
rides.select(
    [
        "VendorID",
        "tpep_pickup_datetime",
        "passenger_count",
        "trip_distance",
        "payment_type",
        "total_amount",
    ]
).head(5)

## Step 2 - Filter and sort rides

Batch processing often starts with a simple question such as: "Which rides were expensive enough to stand out?".  
Polars lets us express that with a short pipeline.


In [ ]:
expensive_rides = (
    rides.select(
        [
            "VendorID",
            "passenger_count",
            "trip_distance",
            "payment_type",
            "total_amount",
        ]
    )
    .filter(pl.col("total_amount") >= 40)
    .sort("total_amount", descending=True)
)

expensive_rides.head(10)

## Step 3 - Build a grouped summary

Grouping turns many detailed events into a smaller batch report. Here we summarize the rides by vendor.


In [ ]:
summary_by_vendor = (
    rides.group_by("VendorID")
    .agg(
        pl.len().alias("rides"),
        pl.col("total_amount").mean().round(2).alias("avg_total_amount"),
        pl.col("trip_distance").mean().round(2).alias("avg_trip_distance"),
    )
    .sort("VendorID")
)

summary_by_vendor

## Exercises

- Change the filter so it shows rides between `$40` and `$60`.
- Create a grouped summary by `payment_type`.
- Compare which grouping feels easier to explain to a teammate.


> **Common pitfall:** If a column name is misspelled, Polars raises an error quickly. Use `rides.columns` when you are unsure.


In [ ]:
# TODO: Write your solutions for the exercises above.


<details>
    <summary>Show Reference Solution</summary>

```python
rides_40_to_60 = (
    rides.select(
        [
            "VendorID",
            "passenger_count",
            "trip_distance",
            "payment_type",
            "total_amount",
        ]
    )
    .filter((pl.col("total_amount") >= 40) & (pl.col("total_amount") <= 60))
    .sort("total_amount", descending=True)
)

rides_40_to_60.head(10)

# ----------

summary_by_payment_type = (
    rides.group_by("payment_type")
    .agg(
        pl.len().alias("rides"),
        pl.col("total_amount").mean().round(2).alias("avg_total_amount"),
    )
    .sort("payment_type")
)

summary_by_payment_type
```
</details>
